# ShrimpCount training (Colab)
Runtime > Change runtime type > GPU (T4). Run cells top to bottom.

Before this: labeled data exported from Roboflow/CVAT in **YOLO** format, `data/manifest.csv`, and `data/splits.csv` (from `tools/split_by_batch.py`). Zip the repo's `data/` folder (without huge originals if you can) to Drive as `shrimp_data.zip`, and put the repo in Drive as `shrimp.ai/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/shrimp.ai
!pip -q install ultralytics

In [ ]:
# Build the split-by-batch YOLO dataset (test split is copied but never trained on)
!python training/prepare_yolo_dataset.py --raw data/raw --labels data/labels --out /content/yolo

In [ ]:
# Baseline: nano model, full frames at 1280. Watch val mAP50 but judge by COUNT error below.
!python training/train.py --data /content/yolo/data.yaml --model yolo11n.pt --imgsz 1280 --epochs 100 --project /content/runs

In [ ]:
# Count predictions on the held-out TEST split, then the evaluation report
!python eval/predict_counts.py --weights /content/runs/larva/weights/best.pt --out eval/out/counts.csv
!python eval/evaluate_counts.py eval/out/counts.csv --target-mape 5

In [ ]:
# Save the ONNX model for the API
!cp /content/runs/larva/weights/best.onnx model/best.onnx